In [2]:
import pandas as pd
import numpy as np
import re
import nltk
import pickle
nltk.download("punkt")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from gensim.models import Word2Vec, KeyedVectors

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ashwi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
df = pd.read_csv("D:\\news\\Classfication\\news.tsv", sep="\t")
print(df.head())

  News ID Category         Topic  \
0  N10000   sports        soccer   
1  N10001     news  newspolitics   
2  N10002     news        newsus   
3  N10003     news  newspolitics   
4  N10004     news     newsworld   

                                            Headline  \
0  Predicting Atlanta United's lineup against Col...   
1  Mitch McConnell: DC statehood push is 'full bo...   
2            Home In North Highlands Damaged By Fire   
3  Meghan McCain blames 'liberal media' and 'thir...   
4                            Today in History: Aug 1   

                                           News body  \
0  Only FIVE internationals allowed, count em, FI...   
1  WASHINGTON -- Senate Majority Leader Mitch McC...   
2  NORTH HIGHLANDS (CBS13)   Fire damaged a home ...   
3  Meghan McCain is speaking out after a journali...   
4  1714: George I becomes King Georg Ludwig, Elec...   

                                Title entity  \
0  {"Atlanta United's": 'Atlanta United FC'}   
1            

In [9]:
df["Category"].value_counts()

Category
sports           30557
news             26689
finance          10571
lifestyle         7405
autos             5494
travel            5381
foodanddrink      5286
video             4968
tv                3981
health            3799
weather           3298
music             2547
movies            1996
entertainment     1487
kids               299
europe               2
northamerica         1
adexperience         1
Name: count, dtype: int64

In [10]:
df.isnull().sum()

News ID            0
Category           0
Topic              0
Headline           0
News body         58
Title entity       0
Entity content     0
dtype: int64

In [11]:
# Fill NaN values to avoid errors
df['Headline'] = df.get('Headline', '').fillna('').astype(str)
df['News body'] = df.get('News body', '').fillna('').astype(str)

In [12]:
list(df.columns)

['News ID',
 'Category',
 'Topic',
 'Headline',
 'News body',
 'Title entity',
 'Entity content']

In [13]:
df.isnull().sum()

News ID           0
Category          0
Topic             0
Headline          0
News body         0
Title entity      0
Entity content    0
dtype: int64

In [14]:
print(df.shape)

(113762, 7)


In [15]:
# Combine Headline + News body
df['text'] = (df['Headline'].str.strip() + ' ' + df['News body'].str.strip()).str.strip()
df['text'] = df['text'].fillna('')

# Feature and target
X = df["text"]
y = df["Category"]

In [16]:
for i in range(5):
    print("TEXT:", X.iloc[i])
    print("CATEGORY:", y.iloc[i])
    print("-" * 50)

TEXT: Predicting Atlanta United's lineup against Columbus Crew in the U.S. Open Cup Only FIVE internationals allowed, count em, FIVE! So first off we should say, per our usual Atlanta United lineup predictions, this will be wrong. Why will it be wrong? Well, aside from the obvious, we still don't have a ton of data points from Frank de Boer in how he prefers to rotate his team for   let's be honest   an inferior competition. We've seen how he rotates (or doesn't rotate) in CONCACAF Champions League play, but that's a bit different because CCL was clearly a priority for the club. We got one glimpse of U.S. Open Cup rotation last week when the team played a home game as the visiting team against the Charleston Battery, but will things change on the actual road against an MLS club? Here's my predicted lineup: Let's talk about it: Kann - Seems like he's the Cup keeper. Simples. CBs - I think Leandro Gonzalez Pirez is likely to be a casualty of the 5-international player limit. Miles Robins

In [17]:
# keep only required columns
df = df[["Headline", "Title entity", "Entity content"]]

In [18]:
# Take only 50% of the data to reduce computation
df = df.sample(frac=0.5, random_state=42).reset_index(drop=True)

In [19]:
df.head()

,Headline,Title entity,Entity content
0,Stars who came out,{},{}
1,19 Ice Cream Pies You'll Want to Make All Summ...,{},{}
2,Mixed-used development will bring variety to d...,{},{}
3,Paul Manafort Seemed Headed to Rikers. Then th...,{'Justice Department': 'United States Departme...,{'United States Department of Justice': {'type...
4,Sean Shelby's Shoes: What's next for Junior Do...,"{'Junior Dos Santos': 'Junior dos Santos', 'UF...","{'Junior dos Santos': {'type': 'item', 'id': '..."


In [20]:
import ast

def parse_dict(x):
    try:
        return ast.literal_eval(x)
    except:
        return {}

df["title_entity_dict"] = df["Title entity"].apply(parse_dict)

In [21]:
import re

def clean_text_ner(text):
    text = re.sub(r"<.*?>", "", str(text))
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_text"] = df["Headline"].apply(clean_text_ner)

In [22]:
def tokenize(text):
    return text.split()

df["tokens"] = df["clean_text"].apply(tokenize)

In [23]:
import spacy

In [24]:
import spacy
nlp = spacy.blank("en")  # ✅ No downloads needed - works instantly
df["Headline"] = df["Headline"].astype(str)
df["Title entity"] = df["Title entity"].astype(str)
print("✅ NLP pipeline ready!")


COUNTRIES = ["United States", "India", "Brazil", "China", "Mexico", "Canada"]
PERSON_PATTERN = r"^[A-Z][a-z]+(\s[A-Z][a-z]+)+$"
ORG_KEYWORDS = ["Corporation", "Authority", "Committee", "Association", "University", "Agency", "Company", "FC", "Ltd"]


def infer_entity_type(expanded):
    expanded = expanded.strip()

    if re.match(PERSON_PATTERN, expanded):
        return "PERSON"

    if expanded in COUNTRIES:
        return "LOCATION"

    if any(k in expanded for k in ORG_KEYWORDS):
        return "ORG"

    return "MISC"

def convert_to_bio(text, entity_string):
    tokens = text.split()
    tags = ["O"] * len(tokens)

    if entity_string == "{}":
        return tokens, tags

    try:
        ent_dict = ast.literal_eval(entity_string)
    except:
        return tokens, tags
    lower_tokens = [w.lower().strip(".,!?") for w in tokens]

    for surface, expanded in ent_dict.items():
        clean_surface = surface.replace("'s", "").strip()
        stoks = clean_surface.split()
        stoks = [w.lower().strip(".,!?") for w in stoks]
        n = len(stoks)

        ent_type = infer_entity_type(expanded)

        # Search entity span safely
        for i in range(len(tokens)):
            try:
                if lower_tokens[i:i+n] == stoks:
                    tags[i] = f"B-{ent_type}"
                    for j in range(i+1, i+n):
                        if j < len(tags):   # SAFETY CHECK
                            tags[j] = f"I-{ent_type}"
            except:
                continue

    return tokens, tags



sentences = []
labels = []

for _, row in df.iterrows():
    s, t = convert_to_bio(row["Headline"], row["Title entity"])
    sentences.append(s)
    labels.append(t)

print("DATA READY — Samples:", len(sentences))

✅ NLP pipeline ready!
DATA READY — Samples: 56881


In [25]:
# ENCODING & VOCAB GENERATION

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# Define word_set FIRST (missing from your code)
word_set = set(w for seq in X for w in seq)
tag_set = set(t for seq in labels for t in seq)

# FIXED: Create word2idx from word_set (not word2idx.keys()!)
word2idx = {w: i for i, w in enumerate(word_set)}

word2idx[""] = 0      # Padding token
word2idx["<UNK>"] = 1 # Unknown token


tag2idx = {t: i for i, t in enumerate(sorted(tag_set))}
idx2tag = {v: k for k, v in tag2idx.items()}

MAX_LEN = 60

X = [[word2idx.get(w, 1) for w in seq] for seq in sentences]
X = pad_sequences(X, maxlen=MAX_LEN, padding="post")

y = [[tag2idx[t] for t in seq] for seq in labels]
y = pad_sequences(y, maxlen=MAX_LEN, padding="post")
y_cat = to_categorical(y, num_classes=len(tag2idx))

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y_cat, test_size=0.2, random_state=42
)

VOCAB_SIZE = len(word2idx)
NUM_TAGS = len(tag2idx)

print("VOCAB SIZE:", VOCAB_SIZE)
print("NUM TAGS:", NUM_TAGS)



VOCAB SIZE: 2242
NUM TAGS: 8


In [26]:
max(word2idx.values()), len(word2idx)

(2239, 2242)

In [27]:
#  Word2vec

import numpy as np
from gensim.models import Word2Vec
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=4)

embedding_w2v = np.zeros((VOCAB_SIZE, 100))
for w, i in word2idx.items():
    embedding_w2v[i] = w2v_model.wv[w] if w in w2v_model.wv else np.random.normal(0,0.6,100)

In [28]:
# GloVe

GLOVE_PATH = "D:\\news\\glove.6B.100d.txt"
import numpy as np

glove_index = {}
with open(GLOVE_PATH, encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        glove_index[word] = vector

print("Total embeddings found:", len(glove_index))

# Matrix

embedding_glove = np.zeros((VOCAB_SIZE, 100))

for w, i in word2idx.items():
    embedding_glove[i] = glove_index.get(
        w, np.random.normal(scale=0.6, size=(100,))
    )

Total embeddings found: 400000


Rule-based / Heuristic Baseline:

In [29]:
import re

In [30]:
PATTERNS = {
    "PERSON": re.compile(r"\b([A-Z][a-z]+(?:\s[A-Z][a-z]+)*)\b"),
    
    "DATE": re.compile(
        r"\b(\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|"      # 12/05/2023
        r"\d{4}|"                                # 2023
        r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s\d{1,2},?\s\d{4})\b",
        re.IGNORECASE
    ),

    "ORG": re.compile(
        r"\b([A-Z][a-zA-Z& ]+(?:Inc|Ltd|LLC|Corp|University|Institute))\b"
    ),

    "LOC": re.compile(
        r"\b(New York|London|Paris|India|USA|Germany|China)\b"
    )
}

In [31]:
def rule_based_ner(text):
    entities = []

    for label, pattern in PATTERNS.items():
        for match in pattern.finditer(text):
            entities.append({
                "text": match.group(),
                "label": label,
                "start": match.start(),
                "end": match.end()
            })

    return entities

In [32]:
text = "Barack Obama visited New York on July 4, 2012 and spoke at Harvard University."

entities = rule_based_ner(text)

for e in entities:
    print(e)

{'text': 'Barack Obama', 'label': 'PERSON', 'start': 0, 'end': 12}
{'text': 'New York', 'label': 'PERSON', 'start': 21, 'end': 29}
{'text': 'July', 'label': 'PERSON', 'start': 33, 'end': 37}
{'text': 'Harvard University', 'label': 'PERSON', 'start': 59, 'end': 77}
{'text': 'July 4, 2012', 'label': 'DATE', 'start': 33, 'end': 45}
{'text': 'Harvard University', 'label': 'ORG', 'start': 59, 'end': 77}
{'text': 'New York', 'label': 'LOC', 'start': 21, 'end': 29}


In [33]:
def ner_to_bio(tokens, entities):
    tags = ["O"] * len(tokens)

    for ent in entities:
        ent_tokens = ent["text"].split()
        for i in range(len(tokens)):
            if tokens[i:i+len(ent_tokens)] == ent_tokens:
                tags[i] = f"B-{ent['label']}"
                for j in range(1, len(ent_tokens)):
                    tags[i+j] = f"I-{ent['label']}"

    return list(zip(tokens, tags))

In [34]:
tokens = text.split()
bio_tags = ner_to_bio(tokens, entities)
bio_tags

[('Barack', 'B-PERSON'),
 ('Obama', 'I-PERSON'),
 ('visited', 'O'),
 ('New', 'B-LOC'),
 ('York', 'I-LOC'),
 ('on', 'O'),
 ('July', 'B-DATE'),
 ('4,', 'I-DATE'),
 ('2012', 'I-DATE'),
 ('and', 'O'),
 ('spoke', 'O'),
 ('at', 'O'),
 ('Harvard', 'O'),
 ('University.', 'O')]

In [35]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, TimeDistributed, Dense
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

# ============================================
# Regular BiLSTM Model
# ============================================

def build_bilstm(embedding_matrix):
    inp = Input(shape=(MAX_LEN,))
    emb = Embedding(
        VOCAB_SIZE, 100,
        weights=[embedding_matrix],
        mask_zero=True,
        trainable=False
    )(inp)

    x = Bidirectional(LSTM(128, return_sequences=True))(emb)
    out = TimeDistributed(Dense(NUM_TAGS, activation="softmax"))(x)

    model = Model(inp, out)
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model
def build_bilstm_crf(embedding_matrix):
    inp = Input(shape=(MAX_LEN,))
    emb = Embedding(
        VOCAB_SIZE, 100,
        weights=[embedding_matrix],
        mask_zero=True,
        trainable=False
    )(inp)

    x = Bidirectional(LSTM(128, return_sequences=True))(emb)
    logits = TimeDistributed(Dense(NUM_TAGS))(x)

    # Learnable CRF transition matrix
    transitions = tf.Variable(
        tf.random.uniform(shape=(NUM_TAGS, NUM_TAGS)),
        name="transition_matrix"
    )

    def crf_loss(y_true, y_pred):
        """
        y_true → one-hot target
        y_pred → logits
        """
        y_true_idx = tf.argmax(y_true, axis=-1)  # (batch, seq)

        # emission log probabilities
        log_softmax = tf.nn.log_softmax(y_pred, axis=-1)

        # likelihood of correct token prediction
        token_ll = tf.reduce_sum(
            tf.reduce_sum(
                tf.one_hot(y_true_idx, NUM_TAGS) * log_softmax,
                axis=-1
            ),
            axis=-1
        )

        # transition score
        seq_score = 0.0
        for t in range(MAX_LEN - 1):
            curr = y_true_idx[:, t]
            nxt = y_true_idx[:, t + 1]

            seq_score += tf.gather_nd(
                transitions,
                tf.stack([curr, nxt], axis=1)
            )

        loss = -(token_ll + seq_score)

        return tf.reduce_mean(loss)

    model = Model(inp, logits)
    model.compile(optimizer="adam", loss=crf_loss)

    return model

In [36]:
results = []
models_dict = {}

print("\nTraining MODEL-1: BiLSTM + Word2Vec")
model_1 = build_bilstm(embedding_w2v)
model_1.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=5, batch_size=32, callbacks=[early_stop])
models_dict["1"] = model_1


Training MODEL-1: BiLSTM + Word2Vec


Epoch 1/5


1422/1422 [==============================] - 380s 262ms/step - loss: 0.4167 - accuracy: 0.9067 - val_loss: 0.4130 - val_accuracy: 0.9063
Epoch 2/5
1422/1422 [==============================] - 357s 251ms/step - loss: 0.4080 - accuracy: 0.9074 - val_loss: 0.4128 - val_accuracy: 0.9063
Epoch 3/5
1422/1422 [==============================] - 359s 252ms/step - loss: 0.4075 - accuracy: 0.9074 - val_loss: 0.4114 - val_accuracy: 0.9063
Epoch 4/5
1422/1422 [==============================] - 347s 244ms/step - loss: 0.4069 - accuracy: 0.9074 - val_loss: 0.4115 - val_accuracy: 0.9063
Epoch 5/5
1422/1422 [==============================] - 376s 264ms/step - loss: 0.4066 - accuracy: 0.9074 - val_loss: 0.4113 - val_accuracy: 0.9063


In [37]:
print("\nTraining MODEL-2: BiLSTM + GloVe")
model_2 = build_bilstm(embedding_glove)
model_2.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=5, batch_size=32, callbacks=[early_stop])
models_dict["2"] = model_2


Training MODEL-2: BiLSTM + GloVe
Epoch 1/5
1422/1422 [==============================] - 1491s 1s/step - loss: 0.4173 - accuracy: 0.9068 - val_loss: 0.4134 - val_accuracy: 0.9063
Epoch 2/5
1422/1422 [==============================] - 486s 342ms/step - loss: 0.4082 - accuracy: 0.9074 - val_loss: 0.4133 - val_accuracy: 0.9063
Epoch 3/5
1422/1422 [==============================] - 564s 397ms/step - loss: 0.4075 - accuracy: 0.9074 - val_loss: 0.4122 - val_accuracy: 0.9063
Epoch 4/5
1422/1422 [==============================] - 580s 408ms/step - loss: 0.4068 - accuracy: 0.9074 - val_loss: 0.4114 - val_accuracy: 0.9063
Epoch 5/5
1422/1422 [==============================] - 569s 400ms/step - loss: 0.4065 - accuracy: 0.9074 - val_loss: 0.4113 - val_accuracy: 0.9063


In [38]:
print("\nTraining MODEL-3: BiLSTM-CRF + Word2Vec")
model_3 = build_bilstm_crf(embedding_w2v)
model_3.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=5, batch_size=32, callbacks=[early_stop])
models_dict["3"] = model_3


Training MODEL-3: BiLSTM-CRF + Word2Vec
Epoch 1/5
1422/1422 [==============================] - 719s 494ms/step - loss: 19.3145 - val_loss: -12.4165
Epoch 2/5
1422/1422 [==============================] - 444s 312ms/step - loss: -23.4101 - val_loss: -29.8431
Epoch 3/5
1422/1422 [==============================] - 454s 319ms/step - loss: -32.4365 - val_loss: -34.0942
Epoch 4/5
1422/1422 [==============================] - 445s 313ms/step - loss: -35.0236 - val_loss: -35.5975
Epoch 5/5
1422/1422 [==============================] - 445s 313ms/step - loss: -36.0133 - val_loss: -36.2343


In [39]:
print("\nTraining MODEL-4: BiLSTM-CRF + GloVe")
model_4 = build_bilstm_crf(embedding_glove)
model_4.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=5, batch_size=32, callbacks=[early_stop])
models_dict["4"] = model_4


Training MODEL-4: BiLSTM-CRF + GloVe
Epoch 1/5
1422/1422 [==============================] - 524s 364ms/step - loss: 54.9629 - val_loss: 23.2337
Epoch 2/5
1422/1422 [==============================] - 705s 496ms/step - loss: 12.2345 - val_loss: 5.8104
Epoch 3/5
1422/1422 [==============================] - 501s 352ms/step - loss: 3.2063 - val_loss: 1.5583
Epoch 4/5
1422/1422 [==============================] - 502s 353ms/step - loss: 0.6191 - val_loss: 0.0487
Epoch 5/5
1422/1422 [==============================] - 821s 578ms/step - loss: -0.3733 - val_loss: -0.5906


In [40]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

def evaluate_model(model, X_test, y_test):
    y_prob = model.predict(X_test)
    y_pred = (y_prob > 0.5).astype(int)

    precision = precision_score(y_test, y_pred, average="binary")
    recall    = recall_score(y_test, y_pred, average="binary")
    f1        = f1_score(y_test, y_pred, average="binary")

    return precision, recall, f1

In [41]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "seqeval"])

from seqeval.metrics import precision_score, recall_score, f1_score
import numpy as np

def evaluate_ner_model(model, X_val, y_val, idx2tag):
    pass  # Your function

print("✅ seqeval ready!")


✅ seqeval ready!


In [47]:
from seqeval.metrics import precision_score, recall_score, f1_score
import numpy as np

def evaluate_ner_model(model, X_val, y_val, idx2tag):
    y_pred = model.predict(X_val)
    
    # If CRF → output already decoded
    if len(y_pred.shape) == 2:
        y_pred_ids = y_pred
    else:
        y_pred_ids = np.argmax(y_pred, axis=-1)

    true_tags = []
    pred_tags = []

    for true_seq, pred_seq in zip(y_val, y_pred_ids):
        true_tags.append([idx2tag[t] for t in true_seq if t != 0])
        pred_tags.append([idx2tag[p] for p in pred_seq if p != 0])

    return (
        precision_score(true_tags, pred_tags),
        recall_score(true_tags, pred_tags),
        f1_score(true_tags, pred_tags)
    )

In [48]:
from seqeval.metrics import precision_score, recall_score, f1_score
import numpy as np

def evaluate_ner_model(model, X_val, y_val, idx2tag):
    # Model predictions
    y_pred = model.predict(X_val)

    # Convert predictions to tag indices
    if len(y_pred.shape) == 2:  # CRF decoded output
        y_pred_ids = y_pred
    else:
        y_pred_ids = np.argmax(y_pred, axis=-1)

    # Convert one-hot labels to indices
    if len(y_val.shape) == 3:  # one-hot
        y_true_ids = np.argmax(y_val, axis=-1)
    else:
        y_true_ids = y_val

    true_tags = []
    pred_tags = []

    for true_seq, pred_seq in zip(y_true_ids, y_pred_ids):
        t_tags = []
        p_tags = []

        for t, p in zip(true_seq, pred_seq):
            if t == 0:  # skip PAD
                continue
            t_tags.append(idx2tag[int(t)])
            p_tags.append(idx2tag[int(p)])

        true_tags.append(t_tags)
        pred_tags.append(p_tags)

    return (
        precision_score(true_tags, pred_tags),
        recall_score(true_tags, pred_tags),
        f1_score(true_tags, pred_tags)
    )

In [46]:
results = []

model_info = {
    "1": ("BiLSTM", "Word2Vec"),
    "2": ("BiLSTM", "GloVe"),
    "3": ("BiLSTM-CRF", "Word2Vec"),
    "4": ("BiLSTM-CRF", "GloVe")
}

for key, model in models_dict.items():
    precision, recall, f1 = evaluate_ner_model(
        model, X_val, y_val, idx2tag
    )

    results.append({
        "Model": model_info[key][0],
        "Embedding": model_info[key][1],
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1_Score": round(f1, 4)
    })

356/356 [==============================] - 137s 385ms/step


c:\Users\ashwi\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


356/356 [==============================] - 83s 224ms/step


In [49]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df

,Model,Embedding,Precision,Recall,F1_Score
0,BiLSTM,Word2Vec,0.0,0.0,0.0
1,BiLSTM,GloVe,0.0,0.0,0.0
2,BiLSTM-CRF,Word2Vec,0.0,0.0,0.0
3,BiLSTM-CRF,GloVe,0.0,0.0,0.0


In [50]:
best_row = results_df.loc[results_df["F1_Score"].idxmax()]
best_row

Model          BiLSTM
Embedding    Word2Vec
Precision         0.0
Recall            0.0
F1_Score          0.0
Name: 0, dtype: object

In [51]:
best_model_key = results_df["F1_Score"].idxmax() + 1
best_model = models_dict[str(best_model_key)]
best_model.save("best_ner_model.keras")